# Fit GT trajectory

Open-loop joint replay fails when the live base is not on the demo
`(x, y, yaw)`. This notebook takes a saved world-frame TCP GT
(`extract_GT_trajectory.ipynb`) and a **different** demonstration's
mobile-base pose as a stand-in for that live robot.

- **Step 1:** overlay GT vs live base; resample live `(x, y, yaw)(t)`
  onto GT timestamps.
- **Step 2:** FK of GT arm joints on the live base — the TCP drifts.
- **Step 3:** right-arm IK so TCP tracks the GT from the live base.
- **Step 4:** overlay an eval `tcp_trajectory.csv` (measured 20 Hz FK)
  on the same GT — the check for unexpected live offsets.

Kernel: numpy / matplotlib (lerobot-arena is fine). No ROS, no Isaac.
Set `GT_EPISODE_IDX`, `BASE_EPISODE_IDX`, and (for step 4) `EVAL_DIR`.


### Interpreter and repo path

- Import numpy / matplotlib only — no ROS, no Isaac, no lerobot.
- Pin `ROOT` to the camelo-ebim checkout so dataset paths are absolute.
- Put `ROOT` and `scripts/debugging` on `sys.path`.
- Print `sys.executable` to confirm the kernel (lerobot-arena is fine).
- Why: this notebook is offline. GT comes from an npz, live base from parquet.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("/home/ubuntu/workspace/camelo-ebim")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DEBUG = ROOT / "scripts" / "debugging"
if str(DEBUG) not in sys.path:
    sys.path.insert(0, str(DEBUG))

print(f"python={sys.executable}")
print(f"root={ROOT}")

### Imports and knobs

- Import `camelo.contracts` for the 37-dim state slice `S_BASE_ODOM`.
- Import `gt_traj_utils` (load/plot/resample) and `MobileFR3Kinematics`.
- `RightArmIK` is the step-3 solver (same kinematics object as FK).
- `GT_EPISODE_IDX` / `BASE_EPISODE_IDX` must differ; `EVAL_DIR` is step 4.
- Why: GT npz + a second demo's base stand in for live-base offset.


In [ ]:
from gt_traj_utils import (
    ensure_tabular,
    fmt_xyz_mm,
    load_episode,
    load_gt_traj,
    plot_mobile_base_overlay,
    resample_xy_yaw,
    resample_xyz,
    wrap_pi,
)

from camelo import contracts as C
from camelo.control.kinematics import (
    RIGHT_TCP,
    MobileFR3Kinematics,
    RightArmIK,
)
from camelo.policy.adapters.replay import (
    DEFAULT_ACTIONS_DIR,
    DEFAULT_DATASET_DIR,
    DEFAULT_REPO_ID,
)

# --- knobs ---
GT_EPISODE_IDX = 89
BASE_EPISODE_IDX = 1
REPO_ID = DEFAULT_REPO_ID
DATASET_DIR = ROOT / DEFAULT_DATASET_DIR
OUT_DIR = ROOT / DEFAULT_ACTIONS_DIR
PLOT_EVERY_N = 20
HEADING_TICK_M = 0.15
EVAL_DIR = ROOT / "outputs" / "eval" / "20260814_164340"  # the run folder
EVAL_EPISODE = 0

if GT_EPISODE_IDX == BASE_EPISODE_IDX:
    raise SystemExit(
        "GT_EPISODE_IDX and BASE_EPISODE_IDX must differ — the live base "
        "is a stand-in from another demonstration"
    )

GT_PATH = OUT_DIR / f"ep{GT_EPISODE_IDX:03d}_gt_traj.npz"
print(f"repo={REPO_ID}")
print(f"gt_episode={GT_EPISODE_IDX}  base_episode={BASE_EPISODE_IDX}")
print(f"gt_npz={GT_PATH}")
print(f"dataset={DATASET_DIR}")
print(f"eval_dir={EVAL_DIR}  eval_episode={EVAL_EPISODE}")


### Load saved GT trajectory

- Read `epXXX_gt_traj.npz` written by `extract_GT_trajectory.ipynb`.
- Required keys: `t`, `base_xy_yaw`, `spine`, `arm_q`, TCP pose, `fps`.
- Do not re-run FK here — `tcp_xyz` in the file is the GT for later IK.
- Fail with a pointer to the extract notebook if the npz is missing.
- Print duration and frame-0 base so the file matches the intended episode.


In [ ]:
gt = load_gt_traj(GT_PATH)
t_gt = np.asarray(gt["t"], dtype=np.float64)
base_gt = np.asarray(gt["base_xy_yaw"], dtype=np.float64)
spine_gt = np.asarray(gt["spine"], dtype=np.float64)
arm_q_gt = np.asarray(gt["arm_q"], dtype=np.float64)
tcp_xyz_gt = np.asarray(gt["tcp_xyz"], dtype=np.float64)
fps_gt = float(np.asarray(gt["fps"]))
print(
    f"GT episode={int(gt['episode'])}  T={len(t_gt)}  "
    f"fps={fps_gt:g}  duration={t_gt[-1] - t_gt[0]:.1f} s"
)
print(
    f"GT base0 x={base_gt[0, 0]:.4f} y={base_gt[0, 1]:.4f} "
    f"yaw={np.degrees(base_gt[0, 2]):+.4f} deg"
)
print(
    f"arm_q={arm_q_gt.shape}  tcp={tcp_xyz_gt.shape}  "
    f"spine0={float(spine_gt[0]):.4f} m"
)


### Load live-base episode from the dataset

- Reuse the same cached parquet as extract (`ensure_tabular`).
- Load `BASE_EPISODE_IDX` only — this is the displaced live `(x, y, yaw)`.
- Time `t_live = frame_index / fps`; slice `S_BASE_ODOM` from state.
- Keep native length; resampling onto GT time happens in the next cell.
- Why: a second demo is a stand-in for control noise at the start pose.


In [ ]:
ensure_tabular(REPO_ID, DATASET_DIR)
ep_live, fps_live = load_episode(DATASET_DIR, BASE_EPISODE_IDX)
state_live = np.stack(
    [np.asarray(row, dtype=np.float64) for row in ep_live["observation.state"]]
)
if state_live.shape[1] != C.STATE_DIM:
    raise ValueError(f"expected state dim {C.STATE_DIM}, got {state_live.shape}")
t_live = ep_live["frame_index"].to_numpy(dtype=np.float64) / fps_live
base_live = state_live[:, C.S_BASE_ODOM].copy()
print(
    f"live episode={BASE_EPISODE_IDX}  T={len(t_live)}  "
    f"fps={fps_live:g}  duration={t_live[-1] - t_live[0]:.1f} s"
)
print(
    f"live base0 x={base_live[0, 0]:.4f} y={base_live[0, 1]:.4f} "
    f"yaw={np.degrees(base_live[0, 2]):+.4f} deg"
)


### Overlay GT vs live mobile-base pose

- Two colors on the same 4-panel plot (`plot_mobile_base_overlay`).
- Each series uses its native `t` (episodes may differ in length).
- Resample live `(x, y, yaw)` onto GT timestamps (clamp; unwrap yaw).
- Log frame-0 and median `Δx, Δy, Δxy` (mm, 4 decimals) and `Δyaw` (deg).
- Why: this displacement is why naive joint replay cannot track the GT TCP.


In [ ]:
fig_b, _axes = plot_mobile_base_overlay(
    [
        (t_gt, base_gt, "tab:blue", f"GT ep{GT_EPISODE_IDX:03d}"),
        (t_live, base_live, "tab:orange", f"live ep{BASE_EPISODE_IDX:03d}"),
    ],
    title=(
        f"mobile base pose — GT ep{GT_EPISODE_IDX:03d} vs "
        f"live ep{BASE_EPISODE_IDX:03d}"
    ),
    plot_every_n=PLOT_EVERY_N,
    heading_tick_m=HEADING_TICK_M,
)
plt.show()

base_live_on_gt = resample_xy_yaw(t_live, base_live, t_gt)
dxy = base_live_on_gt[:, :2] - base_gt[:, :2]
dyaw = wrap_pi(base_live_on_gt[:, 2] - base_gt[:, 2])
dxy_norm = np.hypot(dxy[:, 0], dxy[:, 1])
print(
    f"aligned live-on-GT  T={len(t_gt)}  "
    f"(clamped if live episode is shorter/longer)"
)
print(
    f"frame 0  Δx={dxy[0, 0] * 1000:+.4f} mm  "
    f"Δy={dxy[0, 1] * 1000:+.4f} mm  "
    f"Δxy={dxy_norm[0] * 1000:.4f} mm  "
    f"Δyaw={np.degrees(dyaw[0]):+.4f} deg"
)
print(
    f"median   Δx={np.median(dxy[:, 0]) * 1000:+.4f} mm  "
    f"Δy={np.median(dxy[:, 1]) * 1000:+.4f} mm  "
    f"Δxy={np.median(dxy_norm) * 1000:.4f} mm  "
    f"Δyaw={np.degrees(np.median(dyaw)):+.4f} deg"
)


### Naive joint replay on the live base

- Replay GT `arm_q` and `spine` with the resampled live `(x, y, yaw)`.
- FK `right_tcp` via `MobileFR3Kinematics` (URDF here; Lula if Isaac exists).
- Same measured joints as the extract npz — not commanded `action`.
- 3D path + xyz vs time + `|naive − GT|` in mm. Start green, end gold.
- Why: those joints were true only on the demo base; the TCP should drift.


In [ ]:
kin = MobileFR3Kinematics()
tcp_naive, _naive_quat = kin.fk_traj(
    arm_q_gt, base_live_on_gt, spine_gt, frame=RIGHT_TCP
)
pos_err = np.linalg.norm(tcp_naive - tcp_xyz_gt, axis=1)
median_mm = float(np.median(pos_err) * 1000)
p95_mm = float(np.percentile(pos_err, 95) * 1000)
max_mm = float(pos_err.max() * 1000)

print("naive FK: GT arm_q + GT spine + live base (resampled)")
print(f"  GT TCP     {fmt_xyz_mm(tcp_xyz_gt[0])}")
print(f"  naive TCP  {fmt_xyz_mm(tcp_naive[0])}")
print(f"  frame 0 Δ  {fmt_xyz_mm(tcp_naive[0] - tcp_xyz_gt[0])}")
print(
    f"  |naive−GT|  median={median_mm:.4f} mm  "
    f"p95={p95_mm:.4f} mm  max={max_mm:.4f} mm"
)

fig_n = plt.figure(figsize=(16, 5.2))
ax3 = fig_n.add_subplot(1, 3, 1, projection="3d")
ax3.plot(*tcp_xyz_gt.T, color="tab:blue", lw=1.4, label="GT TCP")
ax3.plot(*tcp_naive.T, color="tab:orange", lw=1.2, label="naive FK")
ax3.scatter(*tcp_xyz_gt[0], c="green", s=30, marker="o", zorder=5)
ax3.scatter(*tcp_xyz_gt[-1], c="gold", s=30, marker="o", zorder=5)
ax3.scatter(*tcp_naive[0], c="green", s=30, marker="s", zorder=5)
ax3.scatter(*tcp_naive[-1], c="gold", s=30, marker="s", zorder=5)
ax3.set_xlabel("x (m)")
ax3.set_ylabel("y (m)")
ax3.set_zlabel("z (m)")
ax3.set_title("right TCP world path")
ax3.legend(loc="best", fontsize=8)

ax_xyz = fig_n.add_subplot(1, 3, 2)
for arr, label, color in (
    (tcp_xyz_gt, "GT", "tab:blue"),
    (tcp_naive, "naive", "tab:orange"),
):
    ax_xyz.plot(t_gt, arr[:, 0], color=color, lw=1.0, label=f"{label} x")
    ax_xyz.plot(t_gt, arr[:, 1], color=color, lw=1.0, ls="--")
    ax_xyz.plot(t_gt, arr[:, 2], color=color, lw=1.0, ls=":")
ax_xyz.set_xlabel("t (s)")
ax_xyz.set_ylabel("m")
ax_xyz.set_title("TCP x / y / z vs time")
ax_xyz.legend(loc="best", fontsize=7, ncol=2)
ax_xyz.grid(True, alpha=0.3)

ax_err = fig_n.add_subplot(1, 3, 3)
ax_err.plot(t_gt, pos_err * 1000, color="tab:red", lw=1.0)
ax_err.set_xlabel("t (s)")
ax_err.set_ylabel("mm")
ax_err.set_title("|naive TCP − GT TCP|")
ax_err.grid(True, alpha=0.3)

fig_n.suptitle(
    f"naive joint replay on live base — GT ep{GT_EPISODE_IDX:03d} / "
    f"live ep{BASE_EPISODE_IDX:03d}"
)
fig_n.tight_layout()
plt.show()


### Right-arm IK on the live base

- Solve `RightArmIK` for world GT TCP xyz given the resampled live base.
- Spine stays the measured GT height; orientation is not tracked.
- Frame 0 seeds from GT `arm_q[0]`; later frames seed from the previous q
  (perfect joint execution).
- FK that `arm_q` on the live base; overlay GT / naive / IK in 3D.
- Why: joints that account for the base offset should put TCP on the GT path.


In [ ]:
ik = RightArmIK(kin)
arm_q_ik, ik_ok = ik.solve_traj(
    tcp_xyz_gt, base_live_on_gt, spine_gt, arm_q_gt[0]
)
tcp_ik, _ik_quat = kin.fk_traj(
    arm_q_ik, base_live_on_gt, spine_gt, frame=RIGHT_TCP
)
ik_err = np.linalg.norm(tcp_ik - tcp_xyz_gt, axis=1)
n_fail = int((~ik_ok).sum())


def _mm_stats(err: np.ndarray) -> tuple[float, float, float]:
    return (
        float(np.median(err) * 1000),
        float(np.percentile(err, 95) * 1000),
        float(err.max() * 1000),
    )


ik_med, ik_p95, ik_max = _mm_stats(ik_err)
nv_med, nv_p95, nv_max = _mm_stats(pos_err)

print(f"IK backend={ik.backend}  T={len(t_gt)}  fails={n_fail}/{len(t_gt)}")
print(f"  GT TCP     {fmt_xyz_mm(tcp_xyz_gt[0])}")
print(f"  naive TCP  {fmt_xyz_mm(tcp_naive[0])}")
print(f"  IK TCP     {fmt_xyz_mm(tcp_ik[0])}")
print(f"  frame 0 Δ IK  {fmt_xyz_mm(tcp_ik[0] - tcp_xyz_gt[0])}")
print(
    f"  |IK−GT|     median={ik_med:.4f} mm  "
    f"p95={ik_p95:.4f} mm  max={ik_max:.4f} mm"
)
print(
    f"  |naive−GT|  median={nv_med:.4f} mm  "
    f"p95={nv_p95:.4f} mm  max={nv_max:.4f} mm"
)

fig_ik = plt.figure(figsize=(16, 5.2))
ax3 = fig_ik.add_subplot(1, 3, 1, projection="3d")
paths = (
    (tcp_xyz_gt, "GT TCP", "tab:blue", "o"),
    (tcp_naive, "naive FK", "tab:orange", "s"),
    (tcp_ik, "IK+FK", "tab:purple", "D"),
)
for arr, label, color, marker in paths:
    ax3.plot(*arr.T, color=color, lw=1.3, label=label)
    ax3.scatter(*arr[0], c="green", s=30, marker=marker, zorder=5)
    ax3.scatter(*arr[-1], c="gold", s=30, marker=marker, zorder=5)
ax3.set_xlabel("x (m)")
ax3.set_ylabel("y (m)")
ax3.set_zlabel("z (m)")
ax3.set_title("right TCP world path")
ax3.legend(loc="best", fontsize=8)

ax_xyz = fig_ik.add_subplot(1, 3, 2)
for arr, label, color in (
    (tcp_xyz_gt, "GT", "tab:blue"),
    (tcp_naive, "naive", "tab:orange"),
    (tcp_ik, "IK", "tab:purple"),
):
    ax_xyz.plot(t_gt, arr[:, 0], color=color, lw=1.0, label=f"{label} x")
    ax_xyz.plot(t_gt, arr[:, 1], color=color, lw=1.0, ls="--")
    ax_xyz.plot(t_gt, arr[:, 2], color=color, lw=1.0, ls=":")
ax_xyz.set_xlabel("t (s)")
ax_xyz.set_ylabel("m")
ax_xyz.set_title("TCP x / y / z vs time")
ax_xyz.legend(loc="best", fontsize=7, ncol=3)
ax_xyz.grid(True, alpha=0.3)

ax_err = fig_ik.add_subplot(1, 3, 3)
ax_err.plot(t_gt, pos_err * 1000, color="tab:red", lw=1.0, label="naive")
ax_err.plot(t_gt, ik_err * 1000, color="tab:purple", lw=1.0, label="IK")
ax_err.set_xlabel("t (s)")
ax_err.set_ylabel("mm")
ax_err.set_title("|TCP − GT TCP|")
ax_err.legend(loc="best", fontsize=8)
ax_err.grid(True, alpha=0.3)

fig_ik.suptitle(
    f"IK vs naive on live base — GT ep{GT_EPISODE_IDX:03d} / "
    f"live ep{BASE_EPISODE_IDX:03d}"
)
fig_ik.tight_layout()
plt.show()


### Overlay eval `tcp_trajectory.csv` vs GT

- Load `EVAL_DIR / episode_NNN / tcp_trajectory.csv` (20 Hz FK of measured
  joints on the ROS/eval client — not the policy-side IK command).
- Resample logged `(x, y, z)(t)` onto GT timestamps (clamp; same idea as
  `resample_xy_yaw`).
- 3-panel: 3D GT TCP (blue) vs live TCP (green); xyz vs time; `|live − GT|`
  mm. Print frame-0 and median / p95 / max offset (mm, 4 decimals).
- Why: unexpected offsets (time lag, clamp, residual vs the offline purple
  IK curve) show up here. Skip if the CSV is missing so steps 1–3 still run.


In [ ]:
import csv

csv_path = EVAL_DIR / f"episode_{EVAL_EPISODE:03d}" / "tcp_trajectory.csv"
if not csv_path.is_file():
    print(f"skip step 4: no eval TCP log at {csv_path}")
    print("set EVAL_DIR to an outputs/eval/<run> folder after a scored eval")
else:
    with csv_path.open(newline="") as f:
        rows = list(csv.DictReader(f))
    t_log = np.array([float(r["t"]) for r in rows], dtype=np.float64)
    xyz_log = np.column_stack(
        [
            np.array([float(r["x"]) for r in rows], dtype=np.float64),
            np.array([float(r["y"]) for r in rows], dtype=np.float64),
            np.array([float(r["z"]) for r in rows], dtype=np.float64),
        ]
    )
    tcp_live = resample_xyz(t_log, xyz_log, t_gt)
    live_err = np.linalg.norm(tcp_live - tcp_xyz_gt, axis=1)
    live_med = float(np.median(live_err) * 1000)
    live_p95 = float(np.percentile(live_err, 95) * 1000)
    live_max = float(live_err.max() * 1000)

    print(f"eval TCP log {csv_path}  n={len(rows)}")
    print(f"  GT TCP     {fmt_xyz_mm(tcp_xyz_gt[0])}")
    print(f"  live TCP   {fmt_xyz_mm(tcp_live[0])}")
    print(f"  frame 0 Δ  {fmt_xyz_mm(tcp_live[0] - tcp_xyz_gt[0])}")
    print(
        f"  |live−GT|  median={live_med:.4f} mm  "
        f"p95={live_p95:.4f} mm  max={live_max:.4f} mm"
    )

    fig_live = plt.figure(figsize=(16, 5.2))
    ax3 = fig_live.add_subplot(1, 3, 1, projection="3d")
    for arr, label, color, marker in (
        (tcp_xyz_gt, "GT TCP", "tab:blue", "o"),
        (tcp_live, "live TCP", "tab:green", "s"),
    ):
        ax3.plot(*arr.T, color=color, lw=1.3, label=label)
        ax3.scatter(*arr[0], c="green", s=30, marker=marker, zorder=5)
        ax3.scatter(*arr[-1], c="gold", s=30, marker=marker, zorder=5)
    ax3.set_xlabel("x (m)")
    ax3.set_ylabel("y (m)")
    ax3.set_zlabel("z (m)")
    ax3.set_title("right TCP world path")
    ax3.legend(loc="best", fontsize=8)

    ax_xyz = fig_live.add_subplot(1, 3, 2)
    for arr, label, color in (
        (tcp_xyz_gt, "GT", "tab:blue"),
        (tcp_live, "live", "tab:green"),
    ):
        ax_xyz.plot(t_gt, arr[:, 0], color=color, lw=1.0, label=f"{label} x")
        ax_xyz.plot(t_gt, arr[:, 1], color=color, lw=1.0, ls="--")
        ax_xyz.plot(t_gt, arr[:, 2], color=color, lw=1.0, ls=":")
    ax_xyz.set_xlabel("t (s)")
    ax_xyz.set_ylabel("m")
    ax_xyz.set_title("TCP x / y / z vs time")
    ax_xyz.legend(loc="best", fontsize=7, ncol=2)
    ax_xyz.grid(True, alpha=0.3)

    ax_err = fig_live.add_subplot(1, 3, 3)
    ax_err.plot(t_gt, live_err * 1000, color="tab:green", lw=1.0, label="live")
    ax_err.set_xlabel("t (s)")
    ax_err.set_ylabel("mm")
    ax_err.set_title("|live TCP − GT TCP|")
    ax_err.legend(loc="best", fontsize=8)
    ax_err.grid(True, alpha=0.3)

    fig_live.suptitle(
        f"eval TCP vs GT — ep{GT_EPISODE_IDX:03d} / eval episode {EVAL_EPISODE:03d}"
    )
    fig_live.tight_layout()
    plt.show()
